# Week 7: Are the Giller nominees concentrated in the big cities?

This notebook was section 5 of the Week 7 notes (`W07_notes.ipynb`). It uses variables created in sections 1 to 4 of those notes (`nominees`, `birthplaces`, `residences`, `born_regions`, `home_regions`, `canadian_regions`, and `get_region()`), so it won't run on its own: run sections 1 to 4 of `W07_notes.ipynb` first, or copy this section back into those notes.

## 5. Are the nominees concentrated in the big cities?

Literary life in Canada (publishers, agents, magazines, festivals, newspapers) is concentrated in a few cities, especially Toronto. Does the prize reflect that? Let's compare where nominees live with where Canadians in general live.

### Step 1: Getting the city

The city is the **first** piece of a place, so we use index `0` where `get_region()` used index `-1`.

In [1]:
def get_city(place):
    """ Return the city at the start of place.

        Example: get_city("Colombo, Sri Lanka") returns "Colombo"
    """
    parts = place.split(",")
    city = parts[0].strip()

    return city


print(get_city("Colombo, Sri Lanka"))
print(get_city("London, ON"))
print(get_city("London, England"))

Colombo
London
London


Uh oh. A city name alone isn't enough: eight nominees were born in a "London", four in Ontario and four in England! To tell them apart we'll keep the city **and** the region together, in the form `"London, ON"`. We already have each author's cleaned region in `home_regions`, so we only need to put the city in front of it.

### Step 2: How many nominees living in Canada live in a big city?

We'll define the "big urban centres" as Canada's three largest cities. Since we are only asking about Canada, we only keep the authors who live in Canada. We build a new dictionary, `home_cities`, that maps each of these authors to their city and region.

In [2]:
home_cities = {}

for author in residences:
    if home_regions[author] in canadian_regions:
        parts = residences[author].split(" and ")
        city = get_city(parts[0])
        home_cities[author] = city + ", " + home_regions[author]

lives_in_canada = len(home_cities)
print(lives_in_canada, "nominees living in Canada")

author = "Dany Laferrière, translated by David Homel"
print(residences[author], "->", home_cities[author])

NameError: name 'residences' is not defined

In [ ]:
big_cities = ["Toronto, ON", "Montreal, QC", "Vancouver, BC"]

lives_in_big_city = 0

for author in home_cities:
    if home_cities[author] in big_cities:
        lives_in_big_city = lives_in_big_city + 1

print(f"{lives_in_big_city} of {lives_in_canada} nominees living in Canada live in a big city ({lives_in_big_city / lives_in_canada * 100:.0f}%)")

Before we trust that number, let's look at every Canadian city in the data along with how many nominees live there. Can you spot residences that *should* have been counted but weren't?

In [ ]:
city_counts = {}

for author in home_cities:
    city = home_cities[author]
    if city in city_counts:
        city_counts[city] = city_counts[city] + 1
    else:
        city_counts[city] = 1

for city in city_counts:
    print(city, city_counts[city])

We missed some because of **city boundaries**. `Scarborough` has been part of the City of Toronto since 1998, but it is recorded as its own place. `West Vancouver` and `New Westminster` are separate cities, but part of the Vancouver area. Is someone who lives in Scarborough a Toronto writer? Is someone from Mississauga?

(A spelling mistake like `"Montrea, QC"` would make us miss an author too, which is one more reason to look at the list rather than trusting the number. The best place to fix spelling mistakes is the spreadsheet itself.)

City boundaries are a real decision. To compare our numbers with population figures in a fair way, we will use the same boundaries that Statistics Canada uses for its **census metropolitan areas**: a big city together with its surrounding suburbs.

### Step 3: Recording our decisions in a dictionary

This is the same pattern as `corrections`, and we use it the same way: the key is the place as it appears in our data, and the value is the big city we count it as.

In [ ]:
metro_areas = {
    # Suburbs inside the Toronto census metropolitan area
    "Scarborough, ON": "Toronto, ON",
    "Willowdale, ON": "Toronto, ON",
    "Mississauga, ON": "Toronto, ON",
    "Brampton, ON": "Toronto, ON",
    "Markham, ON": "Toronto, ON",
    "Newmarket, ON": "Toronto, ON",
    "Richmond Hill, ON": "Toronto, ON",
    # Suburbs inside the Montreal and Vancouver census metropolitan areas
    "Rosemère, QC": "Montreal, QC",
    "West Vancouver, BC": "Vancouver, BC",
    "New Westminster, BC": "Vancouver, BC",
}

# Trying out the dictionary
city = "Scarborough, ON"
if city in metro_areas:
    city = metro_areas[city]
print(city)

(Some of these places, like Willowdale and Markham, only appear in the `Born` column. We include them now because we'll use the same dictionary for birthplaces in Step 5.)

Now we count again. This time, before checking whether a city is in `big_cities`, we look it up in `metro_areas`.

In [ ]:
lives_in_big_city = 0

for author in home_cities:
    city = home_cities[author]
    if city in metro_areas:
        city = metro_areas[city]
    if city in big_cities:
        lives_in_big_city = lives_in_big_city + 1

lives_big_city_percent = lives_in_big_city / lives_in_canada * 100
print(f"Nominees living in Canada who live in a big city: {lives_big_city_percent:.0f}%")

### Step 4: Compared to what?

Is about 55% a lot? A number like this only means something when we compare it with something else. Here we compare it with the share of *all* Canadians who live in these three metropolitan areas. These populations are from Statistics Canada's 2021 Census.

In [ ]:
toronto_population = 6202225
montreal_population = 4291732
vancouver_population = 2642825
canada_population = 36991981

big_city_population = toronto_population + montreal_population + vancouver_population
print(f"Canadians who live in a big city: {big_city_population / canada_population * 100:.0f}%")

### Step 5: Born in a big city, or moved there?

Nominees live in the big three cities much more often than Canadians in general. Were they born there too, or did they move there? To find out, we repeat Steps 1 to 3 for birthplaces: we build `born_cities` for the authors born in Canada, then count how many were born in a big city or one of its suburbs.

In [ ]:
# Steps 1 and 2 again: the city and region of each author born in Canada
born_cities = {}

for author in birthplaces:
    if born_regions[author] in canadian_regions:
        city = get_city(birthplaces[author])
        born_cities[author] = city + ", " + born_regions[author]

# Step 3 again: count the authors born in a big city, including its suburbs
born_in_big_city = 0

for author in born_cities:
    city = born_cities[author]
    if city in metro_areas:
        city = metro_areas[city]
    if city in big_cities:
        born_in_big_city = born_in_big_city + 1

born_big_city_percent = born_in_big_city / len(born_cities) * 100

print(f"Canadian-born nominees born in a big city:         {born_big_city_percent:.0f}%")
print(f"Nominees living in Canada who live in a big city:  {lives_big_city_percent:.0f}%")
print(f"All Canadians (2021 Census):                       {big_city_population / canada_population * 100:.0f}%")

NameError: name 'get_city' is not defined

**Discussion:**
- Nominees born in Canada were born in the big three cities at roughly the same rate as Canadians in general. But more than half of the nominees living in Canada live in one of them. What might explain the move? Think about where publishers, universities, and literary jobs are. (The `Publisher Location` column may help.)
- Our data can't tell us *why* city-dwelling writers are nominated so often. Do writers move to the city to build a career, or are juries and publicists more likely to notice writers who are already there? What other data would help us tell these explanations apart?
- Most nominees were born decades before the 2021 Census, when the big cities held a smaller share of Canada's population. How does that change the comparison between the "born" percentage and the census figure?
- Every decision in our `metro_areas` dictionary changes the result. Would counting Scarborough but not Mississauga be defensible? Should Victoria, Ottawa, or Calgary count as a "big urban centre"? Try changing `big_cities` and see how the result changes.